# ECO375 – Tweet Sentiment & Stock Returns: Data Loading
Source: full_dataset-release.csv (862,231 tweets, 101 firms, 121 dates)

NOTE: the raw CSV contains ~4.2M bare carriage returns (\r) inside unquoted
tweet text, which breaks pd.read_csv. We strip them before parsing.

In [ ]:
import io
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

DATA_PATH = Path(__file__).parent / "full_dataset-release.csv" if "__file__" in globals() else Path("full_dataset-release.csv")


def load_tweets(path=DATA_PATH):
    raw = path.read_bytes()
    raw = raw.replace(b"\r\n", b"\n").replace(b"\r", b" ")
    df = pd.read_csv(io.BytesIO(raw), index_col=0)
    df["DATE"] = pd.to_datetime(df["DATE"], format="%d/%m/%Y")
    df["WEEKDAY"] = df["DATE"].dt.day_name()
    df["IS_WEEKEND"] = df["DATE"].dt.dayofweek >= 5
    return df


df = load_tweets()
assert len(df) == 862_231, f"unexpected row count {len(df)}"
print(df.shape)

In [ ]:
# Quick look
df.head(10)

In [ ]:
# Structure
df.info()

In [ ]:
# Summary stats
df.describe().T

In [ ]:
# Firms ranked by mentions
firm_rank = (
    df.groupby("STOCK")
    .agg(mentions=("TWEET", "size"), unique_tweets=("TWEET", "nunique"), days=("DATE", "nunique"))
    .sort_values("mentions", ascending=False)
)
firm_rank["share_pct"] = (firm_rank["mentions"] / firm_rank["mentions"].sum() * 100).round(2)
firm_rank

In [ ]:
# Recomputed daily price change
# The dataset's k_DAY_RETURN columns are P(t-k)/P(t) - 1 (backward-looking, inverted),
# not forward returns. Rebuild the true close-to-close change and reconcile it.
# Prices are calendar-day: weekends carry Friday's close, so weekend changes are 0.

# Implied close on t-1 (calendar day), recovered from the dataset's own 1_DAY_RETURN
df["PREV_CLOSE"] = df["LAST_PRICE"] * (1 + df["1_DAY_RETURN"])
# True daily price change, close(t-1) -> close(t)
df["DAILY_PRICE_CHG"] = df["LAST_PRICE"] / df["PREV_CLOSE"] - 1

# Independent check: the firm's observed LAST_PRICE on t-1, where that date is in the data
closes = df.groupby(["STOCK", "DATE"])["LAST_PRICE"].first()
prev_key = pd.MultiIndex.from_arrays([df["STOCK"], df["DATE"] - pd.Timedelta(days=1)])
df["DAILY_PRICE_CHG_OBS"] = df["LAST_PRICE"] / closes.reindex(prev_key).to_numpy() - 1
df["RECON_DIFF"] = df["DAILY_PRICE_CHG"] - df["DAILY_PRICE_CHG_OBS"]

# Next-trading-day change, close(t) -> close(next weekday); weekend tweets map to Monday
next_bday = {d: d + pd.offsets.BDay(1) for d in pd.DatetimeIndex(df["DATE"].unique())}
daily_chg = df.groupby(["STOCK", "DATE"])["DAILY_PRICE_CHG"].first()
next_key = pd.MultiIndex.from_arrays([df["STOCK"], df["DATE"].map(next_bday)])
df["NEXT_DAY_PRICE_CHG"] = daily_chg.reindex(next_key).to_numpy()

recon = df["RECON_DIFF"].dropna()
print(f"Reconciled rows: {len(recon):,} | max |diff|: {recon.abs().max():.2e} | "
      f"match within 1e-9: {(recon.abs() < 1e-9).mean():.1%}")
print("Coverage by year:")
print(df.groupby(df["DATE"].dt.year)[["DAILY_PRICE_CHG", "DAILY_PRICE_CHG_OBS", "NEXT_DAY_PRICE_CHG"]]
      .apply(lambda g: g.notna().mean().round(3)))
df[["STOCK", "DATE", "WEEKDAY", "LAST_PRICE", "PREV_CLOSE", "1_DAY_RETURN",
    "DAILY_PRICE_CHG", "DAILY_PRICE_CHG_OBS", "NEXT_DAY_PRICE_CHG"]].drop_duplicates(["STOCK", "DATE"]).query(
    "STOCK == 'Apple' and DATE >= '2018-09-06'").head(10)

In [ ]:
# Firm-day panel (returns/price are constant within firm-day, so aggregate here)
panel = (
    df.groupby(["STOCK", "DATE"])
    .agg(
        n_tweets=("TWEET", "size"),
        lstm_mean=("LSTM_POLARITY", "mean"),
        textblob_mean=("TEXTBLOB_POLARITY", "mean"),
        last_price=("LAST_PRICE", "first"),
        ret_1d=("1_DAY_RETURN", "first"),
        ret_2d=("2_DAY_RETURN", "first"),
        ret_3d=("3_DAY_RETURN", "first"),
        ret_7d=("7_DAY_RETURN", "first"),
        prev_close=("PREV_CLOSE", "first"),
        daily_price_chg=("DAILY_PRICE_CHG", "first"),
        next_day_price_chg=("NEXT_DAY_PRICE_CHG", "first"),
        volume=("PX_VOLUME", "first"),
        vol_10d=("VOLATILITY_10D", "first"),
        vol_30d=("VOLATILITY_30D", "first"),
        is_weekend=("IS_WEEKEND", "first"),
    )
    .reset_index()
)
print(panel.shape)
panel.head(10)

In [ ]:
# Positive / negative mention counts per firm-day from LSTM_POLARITY (binary: +1 / -1)
def lstm_mention_counts(df, firm_col="STOCK", date_col="DATE", polarity_col="LSTM_POLARITY"):
    counts = (
        df.assign(pos_mentions=df[polarity_col].eq(1), neg_mentions=df[polarity_col].eq(-1))
        .groupby([firm_col, date_col])[["pos_mentions", "neg_mentions"]]
        .sum()
        .astype(int)
        .reset_index()
    )
    return counts


mentions = lstm_mention_counts(df)
panel = panel.drop(columns=["pos_mentions", "neg_mentions"], errors="ignore").merge(mentions, on=["STOCK", "DATE"], how="left")
assert (panel["pos_mentions"] + panel["neg_mentions"] == panel["n_tweets"]).all()
panel.loc[panel["STOCK"] == "Amazon", ["STOCK", "DATE", "n_tweets", "pos_mentions", "neg_mentions"]].head(10)

In [ ]:
# Net sentiment per firm-day: positive minus negative LSTM mentions
def net_sentiment(panel, pos_col="pos_mentions", neg_col="neg_mentions"):
    return panel[pos_col] - panel[neg_col]


panel["net_sentiment"] = net_sentiment(panel)
panel.loc[panel["STOCK"] == "Amazon", ["STOCK", "DATE", "pos_mentions", "neg_mentions", "net_sentiment"]].head(10)

In [ ]:
# Firms per day
firms_per_day = df.groupby("DATE")["STOCK"].nunique()
firms_per_day.describe()

In [ ]:
# Figure: tweet volume over time (A) and daily return by firm (B)
import matplotlib.dates as mdates
import matplotlib.pyplot as plt

INK, INK_2, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]  # categorical slots 1-5, fixed order
plt.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["Segoe UI", "DejaVu Sans"],
    "axes.edgecolor": AXIS, "axes.labelcolor": INK_2, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.6, "axes.axisbelow": True, "figure.facecolor": "white", "axes.facecolor": "white",
})

daily = panel[panel["DATE"] >= "2018-07-11"]               # daily window; 2017 is month-end only
vol = daily.pivot(index="DATE", columns="STOCK", values="n_tweets").asfreq("D")  # missing dates -> line gaps
top_firms = firm_rank.index[:5].tolist()
top25 = firm_rank.index[:25].tolist()

fig, (ax_a, ax_b) = plt.subplots(2, 1, figsize=(12, 10), sharex=True, gridspec_kw={"hspace": 0.3})

# Panel A — context firms in gray, top 5 highlighted and direct-labelled
for firm in vol.columns.difference(top_firms):
    ax_a.plot(vol.index, vol[firm], color=AXIS, linewidth=0.6, alpha=0.6, zorder=1)
label_y = {}
for firm, color in zip(top_firms, SERIES):
    s = vol[firm]
    ax_a.plot(s.index, s, color=color, linewidth=2, label=firm, zorder=3)
    label_y[firm] = np.log10(s.dropna().iloc[-1])
# Spread end labels so they don't collide (min gap in log10 units)
prev = None
for firm in sorted(label_y, key=label_y.get, reverse=True):
    if prev is not None and prev - label_y[firm] < 0.17:
        label_y[firm] = prev - 0.17
    prev = label_y[firm]
    ax_a.annotate(firm, (vol.index[-1], 10 ** label_y[firm]), xytext=(6, 0), textcoords="offset points",
                  va="center", fontsize=9, color=INK_2)
ax_a.set_yscale("log")
ax_a.set_ylabel("Tweets per day (log scale)")
ax_a.xaxis.set_major_locator(mdates.MonthLocator())
ax_a.xaxis.set_minor_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
ax_a.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax_a.set_xlim(vol.index.min(), vol.index.max() + pd.Timedelta(days=9))
ax_a.legend(loc="lower left", bbox_to_anchor=(0, 1.0), frameon=False, ncol=5, fontsize=9, labelcolor=INK_2)
ax_a.set_title("A. Daily tweet volume by firm, 11 Jul – 31 Oct 2018 (top 5 highlighted; other 96 firms in gray;\n"
               "    breaks = dates missing from the dataset)", loc="left", fontsize=11, color=INK, pad=28)

# Panel B — daily close-to-close return over time, same firms/colors as A.
# Weekdays only (weekend returns are 0 by construction); missing dates -> line gaps.
ret = (daily[~daily["is_weekend"]]
       .pivot(index="DATE", columns="STOCK", values="daily_price_chg").asfreq("B") * 100)
for firm in ret.columns.difference(top_firms):
    ax_b.plot(ret.index, ret[firm], color=AXIS, linewidth=0.6, alpha=0.5, zorder=1)
for firm, color in zip(top_firms, SERIES):
    ax_b.plot(ret.index, ret[firm], color=color, linewidth=1.6, label=firm, zorder=3)
ax_b.axhline(0, color=MUTED, linewidth=1, zorder=2)
ax_b.set_ylabel("Daily return, close-to-close (%)")
ax_b.legend(loc="lower left", bbox_to_anchor=(0, 1.0), frameon=False, ncol=5, fontsize=9, labelcolor=INK_2)
ax_b.set_title("B. Daily stock return by firm, weekdays 11 Jul – 31 Oct 2018 (same firms highlighted; others in gray)",
               loc="left", fontsize=11, color=INK, pad=28)
ax_b.tick_params(axis="x", labelbottom=True)

fig.savefig("fig_volume_and_returns.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Figure: daily positive vs negative LSTM mentions, top 5 firms (small multiples)
# Positive plotted above zero, negative mirrored below; each panel has its own y-scale.
POS, NEG = "#2a78d6", "#e34948"  # diverging pair: blue (positive) / red (negative)

pos = daily.pivot(index="DATE", columns="STOCK", values="pos_mentions").asfreq("D")  # missing dates -> gaps
neg = daily.pivot(index="DATE", columns="STOCK", values="neg_mentions").asfreq("D")

fig, axes = plt.subplots(len(top_firms), 1, figsize=(12, 2.6 * len(top_firms)), sharex=True,
                         gridspec_kw={"hspace": 0.45})
for ax, firm in zip(axes, top_firms):
    p, n = pos[firm], neg[firm]
    ax.fill_between(p.index, 0, p, color=POS, alpha=0.25, linewidth=0)
    ax.fill_between(n.index, 0, -n, color=NEG, alpha=0.25, linewidth=0)
    ax.plot(p.index, p, color=POS, linewidth=1.6, label="Positive")
    ax.plot(n.index, -n, color=NEG, linewidth=1.6, label="Negative")
    ax.axhline(0, color=MUTED, linewidth=1)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{abs(v):,.0f}"))
    share = p.sum() / (p.sum() + n.sum())
    ax.set_title(f"{firm}  ·  {share:.0%} positive overall", loc="left", fontsize=10.5, color=INK)
    # Direct labels on the first panel; the legend carries identity for the rest
    if ax is axes[0]:
        ax.annotate("Positive ↑", (p.index[-1], p.max() * 0.6), xytext=(6, 0), textcoords="offset points",
                    va="center", fontsize=9, color=INK_2)
        ax.annotate("Negative ↓", (n.index[-1], -n.max() * 0.6), xytext=(6, 0), textcoords="offset points",
                    va="center", fontsize=9, color=INK_2)

axes[0].legend(loc="lower left", bbox_to_anchor=(0, 1.18), frameon=False, ncol=2, fontsize=9, labelcolor=INK_2)
axes[-1].xaxis.set_major_locator(mdates.MonthLocator())
axes[-1].xaxis.set_minor_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
axes[-1].set_xlim(pos.index.min(), pos.index.max() + pd.Timedelta(days=9))
fig.supylabel("Tweets per day (positive above 0, negative below)", color=INK_2, fontsize=10, x=0.06)
fig.suptitle("Daily positive vs negative mentions (LSTM polarity), top 5 firms, 11 Jul – 31 Oct 2018\n"
             "y-scales differ by firm; breaks = dates missing from the dataset",
             x=0.125, ha="left", fontsize=11.5, color=INK, y=0.97)

fig.savefig("fig_pos_neg_mentions.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Figure: top firm — mentions, net sentiment and daily return over time (A–C), net sentiment vs return (D–E)
# Separate panels rather than a dual axis: mentions and returns are on different scales.
firm = top_firms[0]
f = daily[daily["STOCK"] == firm].set_index("DATE").asfreq("D")  # missing dates -> gaps
weekday = f.index.dayofweek < 5
ret = f["daily_price_chg"].where(weekday) * 100      # weekend returns are 0 by construction, so drop them
next_ret = f["next_day_price_chg"] * 100             # weekend sentiment maps to Monday's return


def ols(x, y):
    """Slope, t-stat, correlation and N for y = a + b*x (pairwise non-missing)."""
    m = x.notna() & y.notna()
    x, y = x[m].to_numpy(float), y[m].to_numpy(float)
    X = np.column_stack([np.ones_like(x), x])
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta
    se = np.sqrt(resid @ resid / (len(y) - 2) * np.linalg.inv(X.T @ X)[1, 1])
    return {"slope": beta[1], "t": beta[1] / se, "r": np.corrcoef(x, y)[0, 1], "n": len(y), "intercept": beta[0]}


fig = plt.figure(figsize=(12, 13))
gs = fig.add_gridspec(4, 2, height_ratios=[1, 1, 1, 1.25], hspace=0.55, wspace=0.18)
ax_a = fig.add_subplot(gs[0, :])
ax_b = fig.add_subplot(gs[1, :], sharex=ax_a)
ax_c = fig.add_subplot(gs[2, :], sharex=ax_a)
ax_d, ax_e = fig.add_subplot(gs[3, 0]), fig.add_subplot(gs[3, 1])

# A — positive above zero, negative mirrored below
ax_a.fill_between(f.index, 0, f["pos_mentions"], color=POS, alpha=0.25, linewidth=0)
ax_a.fill_between(f.index, 0, -f["neg_mentions"], color=NEG, alpha=0.25, linewidth=0)
ax_a.plot(f.index, f["pos_mentions"], color=POS, linewidth=1.6, label="Positive")
ax_a.plot(f.index, -f["neg_mentions"], color=NEG, linewidth=1.6, label="Negative")
ax_a.axhline(0, color=MUTED, linewidth=1)
ax_a.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{abs(v):,.0f}"))
ax_a.set_ylabel("Tweets per day")
ax_a.legend(loc="lower left", bbox_to_anchor=(0, 1.0), frameon=False, ncol=2, fontsize=9, labelcolor=INK_2)
ax_a.set_title("A. Positive vs negative mentions", loc="left", fontsize=11, color=INK, pad=22)

# B — net sentiment = positive − negative, bar colour by sign
net = f["net_sentiment"]
ax_b.bar(net.index, net, width=0.8, color=np.where(net >= 0, POS, NEG), linewidth=0)
ax_b.axhline(0, color=MUTED, linewidth=1)
ax_b.set_ylabel("Positive − negative")
ax_b.set_title("B. Net sentiment per day", loc="left", fontsize=11, color=INK)

# C — daily close-to-close return, weekdays only; neutral ink so it isn't read as sentiment
ax_c.bar(ret.index, ret, width=0.8, color=INK_2, linewidth=0)
ax_c.axhline(0, color=MUTED, linewidth=1)
ax_c.set_ylabel("Return (%)")
ax_c.set_title("C. Daily stock return, close-to-close (weekdays)", loc="left", fontsize=11, color=INK)
ax_c.xaxis.set_major_locator(mdates.MonthLocator())
ax_c.xaxis.set_minor_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
ax_c.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax_c.set_xlim(f.index.min() - pd.Timedelta(days=1), f.index.max() + pd.Timedelta(days=1))
for ax in (ax_a, ax_b):
    ax.tick_params(axis="x", labelbottom=False)

# D–E — net sentiment vs same-day and next-trading-day return, with OLS fit
fits = {}
for ax, y, label, letter in [(ax_d, ret, "same-day return", "D"), (ax_e, next_ret, "next-trading-day return", "E")]:
    fit = fits[label] = ols(net, y)
    m = net.notna() & y.notna()
    ax.scatter(net[m], y[m], s=30, color=INK_2, alpha=0.6, edgecolor="white", linewidth=1, zorder=3)
    xs = np.linspace(net[m].min(), net[m].max(), 50)
    ax.plot(xs, fit["intercept"] + fit["slope"] * xs, color=INK, linewidth=1.5, zorder=4)
    ax.axhline(0, color=MUTED, linewidth=1)
    ax.axvline(0, color=MUTED, linewidth=1)
    ax.set_xlabel("Net sentiment (positive − negative tweets)")
    ax.set_title(f"{letter}. Net sentiment vs {label}", loc="left", fontsize=11, color=INK)
    ax.text(0.03, 0.05, f"slope = {fit['slope'] * 100:.3f} pp per 100 net tweets\n"
                        f"t = {fit['t']:.2f}   r = {fit['r']:.2f}   N = {fit['n']}",
            transform=ax.transAxes, va="bottom", fontsize=9, color=INK_2)
ax_d.set_ylabel("Return (%)")

fig.suptitle(f"{firm}: tweet sentiment (LSTM) and stock returns, 11 Jul – 31 Oct 2018",
             x=0.125, ha="left", fontsize=12.5, color=INK, y=0.94)
fig.savefig("fig_net_sentiment_vs_return.png", dpi=200, bbox_inches="tight")
plt.show()

pd.DataFrame(fits).T[["slope", "t", "r", "n"]].round(4)